# Coffee17 Semantic OSR v1

Notebook OSR v1: baseline tiga split x tiga seed, HBP negative control, lalu ARPLoss tanpa confusing-sample GAN pada near/medium seed 123. Checkpoint dan report disimpan di Google Drive.

In [ ]:
# 1/5 SETUP REPO + DATASET BERSIH
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')

REPO = Path('/content/coffee-bean-classification')
DATA_ROOT = Path('/content/coffee17-osr-data')
ORIGINAL = DATA_ROOT / 'original'
CLEAN = DATA_ROOT / 'clean'
FOLD = CLEAN / 'folds' / 'fold_1'
PREPARED = Path('/content/coffee17-osr-prepared')
OUTPUT = Path('/content/drive/MyDrive/coffee17-osr-v1')

if not (REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone',
        'https://github.com/ediprin/coffee-bean-classification.git',
        str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)

subprocess.run([
    sys.executable, '-u', '-m',
    'bilinear_lmmd.data.preparation.prepare_coffee17',
    '--output', str(ORIGINAL),
    '--archive', str(DATA_ROOT / 'coffee17.zip'),
    '--seed', '42',
], check=True)

subprocess.run([
    sys.executable, '-u', '-m',
    'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds',
    '--source-root', str(ORIGINAL / 'source'),
    '--output-root', str(CLEAN),
    '--expected-count', '979',
    '--folds', '5',
    '--seed', '42',
    '--validation-ratio', '0.1',
], check=True)

assert (FOLD / 'source' / 'train').is_dir(), f'Dataset gagal disiapkan: {FOLD}'
print('\n=== SIAP ===')
print('GPU      :', __import__('torch').cuda.get_device_name(0))
print('Dataset  :', FOLD)
print('Prepared :', PREPARED, '(lokal, boleh hilang saat reset)')
print('Output   :', OUTPUT, '(Google Drive, persisten)')

In [ ]:
# 2/5 TRAIN / RESUME BASELINE: 3 TIER x 3 SEED
# Jalankan ulang sel 1 lalu sel ini jika runtime reset. best.pt/last.pt tetap di Drive.
cmd = [
    sys.executable, '-u', '-m',
    'bilinear_lmmd.experiments.run_osr_baselines',
    '--data-root', str(FOLD),
    '--prepared-root', str(PREPARED),
    '--output-root', str(OUTPUT),
    '--protocol', 'configs/osr/coffee17_osr_v1.yaml',
    '--config', 'configs/osr/OSR0_efficientnetv2_gap_ce.yaml',
    '--seeds', '42', '123', '2026',
    '--resume',
]
print('MENJALANKAN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 3/5 RINGKASAN AGREGAT BASELINE TIGA SEED
import json
import pandas as pd

summary_path = OUTPUT / 'reports' / 'osr_v1_aggregate.json'
assert summary_path.is_file(), f'Report belum selesai: {summary_path}'
summary = json.loads(summary_path.read_text())
rows = []
for tier, methods in summary['splits'].items():
    for method, metric in methods.items():
        rows.append({
            'tier': tier,
            'score': method.upper(),
            'known_macro_f1': metric['known_macro_f1']['mean'],
            'known_macro_f1_std': metric['known_macro_f1']['std'],
            'oscr': metric['oscr']['mean'],
            'oscr_std': metric['oscr']['std'],
            'auroc': metric['auroc']['mean'],
            'auroc_std': metric['auroc']['std'],
            'fpr95': metric['fpr95']['mean'],
            'fpr95_std': metric['fpr95']['std'],
            'unknown_rejection': metric['unknown_rejection']['mean'],
        })

table = pd.DataFrame(rows)
display(table.style.format({
    'known_macro_f1': '{:.2%}',
    'known_macro_f1_std': '{:.2%}',
    'oscr': '{:.2%}', 'oscr_std': '{:.2%}',
    'auroc': '{:.2%}', 'auroc_std': '{:.2%}',
    'fpr95': '{:.2%}', 'fpr95_std': '{:.2%}',
    'unknown_rejection': '{:.2%}',
}))
print('\nSAVED:', summary_path)

In [ ]:
# 4/5 NEGATIVE CONTROL HBP: NEAR + MEDIUM, SEED 123
# Hanya dua training baru. Seed 42/2026 dijalankan nanti hanya jika gate PASS.
import json
import pandas as pd

hbp_cmd = [
    sys.executable, '-u', '-m',
    'bilinear_lmmd.experiments.run_osr_hbp_screening',
    '--data-root', str(FOLD),
    '--prepared-root', str(PREPARED),
    '--output-root', str(OUTPUT),
    '--seed', '123',
    '--resume',
]
print('MENJALANKAN:', ' '.join(hbp_cmd))
subprocess.run(hbp_cmd, check=True)

decision_path = OUTPUT / 'reports' / 'osr_hbp_failfast_seed123.json'
decision = json.loads(decision_path.read_text())
comparison = pd.DataFrame([
    {
        'tier': tier,
        'decision': row['decision'],
        'delta_auroc': row['delta']['auroc'],
        'delta_oscr': row['delta']['oscr'],
        'delta_known_macro_f1': row['delta']['known_macro_f1'],
    }
    for tier, row in decision['tiers'].items()
])
display(comparison.style.format({
    'delta_auroc': '{:+.2%}',
    'delta_oscr': '{:+.2%}',
    'delta_known_macro_f1': '{:+.2%}',
}))
print('FINAL:', decision['decision'], '-', decision['next_step'])
print('SAVED:', decision_path)

In [ ]:
# 5/5 FAIL-FAST ARPLoss TANPA CONFUSING-SAMPLE GAN
# Jalankan setelah sel setup. Baseline dibaca dari Drive; hanya dua training baru.
import json
import pandas as pd

arpl_cmd = [
    sys.executable, '-u', '-m',
    'bilinear_lmmd.experiments.run_osr_arpl_screening',
    '--data-root', str(FOLD),
    '--prepared-root', str(PREPARED),
    '--output-root', str(OUTPUT),
    '--seed', '123',
    '--resume',
]
print('MENJALANKAN:', ' '.join(arpl_cmd))
subprocess.run(arpl_cmd, check=True)

arpl_path = OUTPUT / 'reports' / 'osr_arpl_failfast_seed123.json'
arpl = json.loads(arpl_path.read_text())
arpl_table = pd.DataFrame([
    {
        'tier': tier,
        'decision': row['decision'],
        'delta_auroc': row['delta']['auroc'],
        'delta_oscr': row['delta']['oscr'],
        'delta_known_macro_f1': row['delta']['known_macro_f1'],
    }
    for tier, row in arpl['tiers'].items()
])
display(arpl_table.style.format({
    'delta_auroc': '{:+.2%}',
    'delta_oscr': '{:+.2%}',
    'delta_known_macro_f1': '{:+.2%}',
}))
print('FINAL:', arpl['decision'], '-', arpl['next_step'])
print('SAVED:', arpl_path)